In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

vector_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "vector_retrieval_results.csv")
graph_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv")
hybrid_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hybrid_retrieval_results.csv")

vector_results.head(), graph_results.head(), hybrid_results.head()

(  query_artist  rank retrieved_artist  similarity_score
 0  Soundgarden     1        Pearl Jam          0.740760
 1  Soundgarden     2          Nirvana          0.711962
 2  Soundgarden     3             Blur          0.684405
 3      Nirvana     1        Pearl Jam          0.754655
 4      Nirvana     2      Soundgarden          0.711962,
   query_artist  rank       retrieved_artist  graph_score evidence_type  \
 0         Blur     1              Pearl Jam            7    shared_tag   
 1         Blur     2  The Smashing Pumpkins            5    shared_tag   
 2         Blur     3                Nirvana            4    shared_tag   
 3     Deftones     1            Soundgarden            2    shared_tag   
 4     Deftones     2                Nirvana            2    shared_tag   
 
                                             evidence  
 0  ['alternative rock', 'grunge', 'rock', 'art ro...  
 1  ['alternative rock', 'grunge', 'rock', 'shoega...  
 2  ['alternative rock', 'grunge', 'n

In [6]:
vector_top1 = vector_results[vector_results["rank"] == 1][
    ["query_artist", "retrieved_artist", "similarity_score"]
].rename(columns={
    "retrieved_artist": "vector_top1",
    "similarity_score": "vector_score"
})

hybrid_top1 = hybrid_results[hybrid_results["hybrid_rank"] == 1][
    ["query_artist", "retrieved_artist", "hybrid_score"]
].rename(columns={
    "retrieved_artist": "hybrid_top1"
})

combined_table = pd.merge(
    vector_top1, 
    hybrid_top1, 
    on="query_artist", 
    how="inner"
)

combined_table

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676
2,Deftones,Blur,0.691868,Soundgarden,0.781798
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780
5,Blur,Pearl Jam,0.760920,Pearl Jam,0.860920


In [7]:
combined_table["top1_changed"] = (
    combined_table["vector_top1"] != combined_table["hybrid_top1"]
)

combined_table

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score,top1_changed
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780,False
5,Blur,Pearl Jam,0.760920,Pearl Jam,0.860920,False


In [8]:
combined_table[combined_table["top1_changed"] == True]

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score,top1_changed
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True


In [9]:
hybrid_top1_evidence = hybrid_results[hybrid_results["hybrid_rank"] == 1][
    [
        "query_artist",
        "retrieved_artist",
        "similarity_score",
        "shared_member",
        "shared_tag",
        "graph_bonus",
        "hybrid_score"
    ]
].rename(columns={
    "retrieved_artist": "hybrid_top1"
})

evaluation_table = combined_table.merge(
    hybrid_top1_evidence,
    on=["query_artist", "hybrid_top1"],
    how="left"
)

evaluation_table

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score_x,top1_changed,similarity_score,shared_member,shared_tag,graph_bonus,hybrid_score_y
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True,0.711962,1.0,6.0,0.300000,1.011961
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True,0.711962,1.0,6.0,0.285714,0.997676
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True,0.681798,0.0,2.0,0.100000,0.781798
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True,0.740760,1.0,4.0,0.257143,0.997903
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780,False,0.696780,0.0,5.0,0.100000,0.796780
5,Blur,Pearl Jam,0.760920,Pearl Jam,0.860920,False,0.760920,0.0,7.0,0.100000,0.860920


In [16]:
def interpret_result(row):
    if row["top1_changed"] and row["shared_member"] > 0:
        return "Hybrid changed the top result due to shared-member graph evidence."
    elif row["top1_changed"] and row["shared_tag"] > 0:
        return "Hybrid changed the top result due to shared-tag graph evidence."
    elif not row["top1_changed"]:
        return "Vector and hybrid agree on the top result."
    else:
        return "Hybrid changed the top result, but graph evidence should be checked."

evaluation_table["interpretation"] = evaluation_table.apply(interpret_result, axis=1)

evaluation_table = evaluation_table[
    [
        "query_artist",
        "vector_top1",
        "vector_score",
        "hybrid_top1",
        "hybrid_score",
        "top1_changed",
        "shared_member",
        "shared_tag",
        "graph_bonus",
        "interpretation"
    ]
]

evaluation_table

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score,top1_changed,shared_member,shared_tag,graph_bonus,interpretation
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True,1.0,6.0,0.300000,Hybrid changed the top result due to shared-me...
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True,1.0,6.0,0.285714,Hybrid changed the top result due to shared-me...
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True,0.0,2.0,0.100000,Hybrid changed the top result due to shared-ta...
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True,1.0,4.0,0.257143,Hybrid changed the top result due to shared-me...
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780,False,0.0,5.0,0.100000,Vector and hybrid agree on the top result.
5,Blur,Pearl Jam,0.760920,Pearl Jam,0.860920,False,0.0,7.0,0.100000,Vector and hybrid agree on the top result.


In [17]:
out_path = PROJECT_ROOT / "data" / "processed" / "retrieval_evaluation_summary.csv"

evaluation_table.to_csv(out_path, index=False)

out_path

WindowsPath('c:/uni/seriousism/reminisGraph/data/processed/retrieval_evaluation_summary.csv')

In [18]:
pd.read_csv(out_path).head()

,query_artist,vector_top1,vector_score,hybrid_top1,hybrid_score,top1_changed,shared_member,shared_tag,graph_bonus,interpretation
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1.011961,True,1.0,6.0,0.300000,Hybrid changed the top result due to shared-me...
1,Nirvana,Pearl Jam,0.754655,Soundgarden,0.997676,True,1.0,6.0,0.285714,Hybrid changed the top result due to shared-me...
2,Deftones,Blur,0.691868,Soundgarden,0.781798,True,0.0,2.0,0.100000,Hybrid changed the top result due to shared-ta...
3,Pearl Jam,Blur,0.760920,Soundgarden,0.997903,True,1.0,4.0,0.257143,Hybrid changed the top result due to shared-me...
4,The Smashing Pumpkins,Blur,0.696780,Blur,0.796780,False,0.0,5.0,0.100000,Vector and hybrid agree on the top result.


#### Phase 10A Conclusion

This notebook evaluates the vector and hybrid retrieval results side by side.

The comparison focuses on top-1 retrieval results for each query artist. The vector retriever ranks artists using embedding cosine similarity, while the hybrid retriever starts from vector similarity and adds graph-based bonuses from shared members and shared tags.

The evaluation shows whether the hybrid retriever changes the top result compared to the vector baseline. When the top result changes because of shared-member evidence, this suggests that graph relationships can surface historical or relational artist connections that vector similarity alone may not prioritize.

In this initial 6-artist dataset, important cases such as Pearl Jam and Nirvana show how hybrid retrieval can promote graph-supported connections. These cases are useful examples for the final project explanation because they directly connect to the research question: when does graph retrieval improve music discovery compared to vector retrieval?

The current evaluation is qualitative and exploratory because the dataset is still small. Future evaluation can use a larger dataset, full non-truncated retrieval candidates, and more formal relevance labels.

The output is saved as `retrieval_evaluation_summary.csv`.